# Lab 2 — Arquitetura profunda da triagem V1 (Korvian)

Notebook do Lab 2 de Deep Learning (CESAR School, Projeto 4 — Neoenergia PE). Bloco profundo:
**triagem V1** = MobileNetV3-Small pré-treinada no ImageNet, com uma espinha dorsal e três
cabeças (cena, defeitos, legível). O código reaproveitável está em `ml/src/mlinsp/`; aqui só
há orquestração, tabelas e gráficos.

**Regras deste notebook**
- As fotos do cliente **não** aparecem em nenhuma saída (nenhuma miniatura): só tabelas e gráficos.
- Partição **por lote**: treino 20/05 + 21/05, validação 22/05, teste 03/07.
- Temperatura e limiares são ajustados **só na validação**; o teste é usado uma vez por modelo, em E2/K3.
- Caminhos configuráveis por variável de ambiente (ver célula 0). Para executar do zero:
  `bash lab-02/executar.sh` na raiz do repositório.

## 0. Configuração

In [ ]:
import json
import math
import os
import platform
import sys
import time
import warnings
from pathlib import Path

T_INICIO = time.time()
LAB = Path.cwd().resolve()                # o notebook roda de dentro de lab-02/
REPO = LAB.parent
sys.path.insert(0, str(REPO / "ml" / "src"))

# ---- caminhos (sobrescreva por variável de ambiente) ----
BASE = Path(os.environ.get("LAB02_BASE", "../Base cliente")).resolve()
ROTULOS_DIR = Path(os.environ.get("LAB02_ROTULOS_DIR", "../rotulador/saida")).resolve()
CACHE = Path(os.environ.get("LAB02_CACHE", "cache")).resolve()            # gitignored
RESULTADOS = Path(os.environ.get("LAB02_RESULTADOS", "resultados")).resolve()

# ---- protocolo ----
SEED = 42
SEMENTES = [0, 1, 2]
RISCO_META = 0.02        # Y: risco máximo nas fotos decididas (justificativa em E2)
COBERTURA_META = 0.30    # X: piso da meta "decidir 30-40% do volume" (doc 03, seção 9.7)
N_FONTE_K2 = 200         # fotos limpas de origem para o rótulo fraco (K2)
CURVA_N = [50, 100, 200] # bônus: tamanhos do treino (+ "todos")

import cv2
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torchvision

from mlinsp import avaliacao, calibracao, dados, degradacao, heuristicas, modelo, particao, seletiva
from mlinsp.dados import CENAS, COM_MEDIDOR, DEFEITOS

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
modelo.semear(SEED)
NT = torch.get_num_threads()
CACHE.mkdir(parents=True, exist_ok=True)
DEVICE = modelo.escolher_device()
RESUMO = {}  # números-chave, preenchidos ao longo do notebook

print(f"python {platform.python_version()} | torch {torch.__version__} | torchvision {torchvision.__version__}"
      f" | opencv {cv2.__version__} | sklearn {sklearn.__version__} | numpy {np.__version__} | pandas {pd.__version__}")
print(f"máquina: {platform.platform()} | {platform.processor()} | device de treino: {DEVICE}")
print(f"pesos ImageNet em cache: {Path(torch.hub.get_dir()) / 'checkpoints'}")
print(f"BASE        = {BASE}  (existe: {BASE.exists()})")
print(f"ROTULOS_DIR = {ROTULOS_DIR}  (existe: {ROTULOS_DIR.exists()})")
print(f"CACHE       = {CACHE}")

## 1. C — Rótulos e partição

### C2/C3. Rótulos consolidados, distribuição e kappa

Os rótulos vêm do `rotulador/` (cena exclusiva; 5 defeitos multirrótulo; legível 0/1).
`rotulador/consolidar.py` junta os CSVs de cada pessoa por maioria e gera
`rotulos_consolidados.csv` (uma linha por foto) e `todos_os_rotulos.csv` (uma linha por foto ×
rotulador, base do kappa). Campo empatado na maioria chega vazio e a foto sai só daquela tarefa.

In [ ]:
manifesto = dados.ler_manifesto(BASE)
rot, todos = dados.ler_rotulos(ROTULOS_DIR)
SINTETICO = (ROTULOS_DIR / "SINTETICO.txt").exists()

print("=" * 90)
print(f"RÓTULOS LIDOS DE: {ROTULOS_DIR}")
if SINTETICO:
    print("!!! ATENÇÃO: RÓTULOS SINTÉTICOS DE DESENVOLVIMENTO (gerados das heurísticas V0). !!!")
    print("!!! Nenhum número abaixo vale como resultado. Rode com os rótulos humanos reais.    !!!")
else:
    print("Rótulos humanos (saída do rotulador/consolidar.py).")
n_rotuladores = todos["rotulador"].nunique() if todos is not None else 0
print(f"fotos rotuladas (consolidadas): {len(rot)} | linhas foto×rotulador: "
      f"{0 if todos is None else len(todos)} | rotuladores: {n_rotuladores}")
print("=" * 90)

faltando = set(rot["nome_arquivo"]) - set(manifesto["arquivo"])
assert not faltando, f"{len(faltando)} fotos rotuladas não estão em BASE: {sorted(faltando)[:3]}"
rot = rot.merge(manifesto.drop(columns=["lote"]), left_on="nome_arquivo", right_on="arquivo", how="left")
print(f"manifesto: {len(manifesto)} fotos em {manifesto['lote'].nunique()} lotes; "
      f"todas as {len(rot)} rotuladas encontradas na base.")
RESUMO["fotos_rotuladas"] = len(rot)
RESUMO["rotuladores"] = int(n_rotuladores)
RESUMO["rotulos_sinteticos"] = SINTETICO

In [ ]:
def distribuicao(df):
    linhas = []
    for c in ["cena"] + DEFEITOS + ["legivel"]:
        if c == "cena":
            vals = CENAS
        else:
            vals = [1]
        for v in vals:
            sel = df[c] == v
            linha = {"classe": c if c == "cena" else c + " = 1", "valor": v if c == "cena" else "",
                     "total": int(sel.sum()), "% do rotulado": 100 * sel.mean()}
            for lote in sorted(df["lote"].unique()):
                linha[lote] = int((sel & (df["lote"] == lote)).sum())
            linhas.append(linha)
    linhas.append({"classe": "empate em algum campo", "valor": "",
                   "total": int(df["empate"].fillna(0).sum()),
                   "% do rotulado": 100 * df["empate"].fillna(0).mean(),
                   **{l: int(df.loc[df["lote"] == l, "empate"].fillna(0).sum()) for l in sorted(df["lote"].unique())}})
    return pd.DataFrame(linhas)

dist = distribuicao(rot)
display(dist)
print("fotos por lote:", rot["lote"].value_counts().sort_index().to_dict())

A tabela acima é a distribuição medida (seção 7 da ficha): ela define os pesos da perda (`pos_weight` = neg/pos por defeito, peso de cena ∝ 1/frequência) e quais classes têm exemplos suficientes para serem avaliadas (mínimo de 3 positivos no treino e na validação; abaixo disso a métrica sai como n/d).

In [ ]:
consolidar = dados.carregar_consolidar(REPO)
if todos is not None and len(todos):
    kappa = dados.kappa_por_classe(todos, consolidar.kappa_cohen, consolidar.CLASSES, consolidar.LIMIAR_KAPPA)
    multi = todos.groupby("nome_arquivo")["rotulador"].nunique()
    print(f"fotos com 2+ rotuladores (às cegas): {(multi >= 2).sum()}")
    display(kappa.fillna("n/d"))
    baixos = kappa[kappa["alerta"].str.startswith("kappa")]
    for _, r in baixos.iterrows():
        print(f"ALERTA: kappa de '{r['classe']}' abaixo de {consolidar.LIMIAR_KAPPA}: a definição está "
              "ambígua — reescrever o esquema (rotulos/esquema.md) e registrar o que mudou.")
    if baixos.empty:
        print(f"Nenhuma classe com kappa < {consolidar.LIMIAR_KAPPA}.")
    RESUMO["kappa"] = {r["classe"]: (None if pd.isna(r["kappa"]) else round(float(r["kappa"]), 3))
                       for _, r in kappa.iterrows()}
    RESUMO["fotos_dupla_rotulagem"] = int((multi >= 2).sum())
else:
    print("todos_os_rotulos.csv ausente: kappa n/d.")

Kappa calculado com a mesma função do `rotulador/consolidar.py` (média ponderada pelo número de fotos em comum de cada par). A coluna *com medidor* restringe às fotos em que os dois viram digital/eletromecânico, porque nas outras a regra do app impõe defeito e legível = 0 e inflaria a concordância.

### C4. Partição por lote e prova numérica de vazamento zero

Treino = 20/05 + 21/05, validação = 22/05, teste = 03/07 (seis semanas depois: o mais próximo
de produção). Depois de atribuir pelo lote, sai da validação toda foto cujo `id_externo` ou
número de medidor já está no treino, e sai do teste o que já está em treino ou validação.

In [ ]:
part, relatorio = particao.particionar(manifesto)
display(relatorio)

meds_csv = dados.medidores_do_csv(BASE)
from itertools import combinations
comum = pd.DataFrame([{"par_de_lotes": f"{a} x {b}", "medidores_em_comum_no_csv (todas as linhas)": len(meds_csv[a] & meds_csv[b])}
                      for a, b in combinations(sorted(meds_csv), 2)])
comum = comum.merge(particao.medidores_em_comum_por_lote(part).rename(
    columns={"medidores_em_comum": "medidores_em_comum_entre_FOTOS"}), on="par_de_lotes")
display(comum)

print("Prova 1 — base inteira (todas as fotos, depois da remoção):")
prova_base = particao.provar_sem_vazamento(part)
display(prova_base)
particao.assert_sem_vazamento(part)

split_por_arquivo = part.set_index("arquivo")["split"]
rot["split"] = rot["nome_arquivo"].map(split_por_arquivo)
print("Prova 2 — subconjunto rotulado:")
prova_rot = particao.provar_sem_vazamento(rot)
display(prova_rot)
particao.assert_sem_vazamento(rot)
print("rotuladas por conjunto:", rot["split"].value_counts(dropna=False).to_dict())
print("OK: zero id_externo e zero número de medidor em comum entre treino, validação e teste "
      "(base inteira e subconjunto rotulado).")
RESUMO["particao"] = relatorio.set_index("conjunto")[["fotos_finais", "removidas_vazamento"]].to_dict("index")
RESUMO["rotuladas_por_conjunto"] = rot["split"].value_counts().to_dict()

Leitura: os medidores que o guia conta em comum entre lotes (4/3/3 nos CSVs de maio) estão em linhas do CSV, não em fotos: entre fotos a interseção já é zero, e a remoção por vazamento não precisa tirar nenhuma foto (se tirasse, o número estaria em `removidas_vazamento`). As fotos órfãs (sem linha no CSV) não têm medidor para comparar; para elas a prova é pelo `id_externo`, que também dá zero.

## 2. B — O modelo profundo

`ModeloTriagem` (`ml/src/mlinsp/modelo.py`): `mobilenet_v3_small(IMAGENET1K_V1).features` →
`AdaptiveAvgPool2d(1)` → vetor de 576 → três cabeças `Dropout(0,2) + Linear`. O `forward`
devolve **logits**; softmax/sigmoide só na inferência, nunca antes da perda.

### B1. Tabela de formas (entrada real 1 × 3 × 480 × 360)

In [ ]:
m = modelo.ModeloTriagem(pretreinado=True)
formas = modelo.tabela_de_formas(m, torch.zeros(1, 3, modelo.ALTURA, modelo.LARGURA))
tab_formas = pd.DataFrame(formas, columns=["camada", "forma da saída"])
tab_formas["forma da saída"] = tab_formas["forma da saída"].astype(str)
display(tab_formas)

A redução total é 32× (480 × 360 → 15 × 12): cada célula do mapa final resume ~32 × 30 px da foto, o suficiente para cena e qualidade global, não para ler dígitos (a leitura recorta antes, seção 4 do guia).

### B2. Parâmetros totais e treináveis

In [ ]:
def linha_param(nome, mod):
    total, treino = modelo.contar_parametros(mod)
    return {"configuração": nome, "total": total, "treináveis": treino, "% treinável": 100 * treino / total}

tab_param = []
m.congelar_extrator(False)
tab_param.append(linha_param("ajuste fino (tudo destravado), 3 cabeças", m))
m.congelar_extrator(True)
tab_param.append(linha_param("extração (extrator congelado), 3 cabeças", m))
m2 = modelo.ModeloTriagem(pretreinado=False, tarefas=("cena", "defeitos"))
tab_param.append(linha_param("ajuste fino, 2 cabeças (cena+defeitos, = esqueleto)", m2))
m2.congelar_extrator(True)
tab_param.append(linha_param("extração, 2 cabeças (cena+defeitos, = esqueleto)", m2))
r18 = torchvision.models.resnet18(weights=None)
tab_param.append(linha_param("alternativa descartada: ResNet-18 (classificador ImageNet)", r18))
tab_param = pd.DataFrame(tab_param)
display(tab_param.style.format({"total": "{:,}", "treináveis": "{:,}", "% treinável": "{:.2f}"}))
RESUMO["parametros"] = {r["configuração"]: {"total": int(r["total"]), "treinaveis": int(r["treináveis"])}
                        for _, r in tab_param.iterrows()}
del m2, r18

Com o extrator congelado sobram só os parâmetros das cabeças (576 × 10 + 10). É essa diferença de duas ordens de grandeza que permite treinar com algumas centenas de rótulos sem decorar o treino.

### B3. Ativação e perda de cada cabeça, com os rótulos reais como prova

In [ ]:
com_med = rot["cena"].isin(COM_MEDIDOR)
n_def = rot[DEFEITOS].fillna(0).sum(axis=1)
n_2def = int((n_def >= 2).sum())
cooc = pd.DataFrame(0, index=DEFEITOS, columns=DEFEITOS)
for a in DEFEITOS:
    for b in DEFEITOS:
        cooc.loc[a, b] = int(((rot[a] == 1) & (rot[b] == 1)).sum())

# o app grava exatamente uma cena por foto×rotulador; conta as linhas fora disso
n_multicena = int((~todos["cena"].isin(CENAS)).sum()) if todos is not None else 0
leg_com_defeito = int(((n_def >= 1) & (rot["legivel"] == 1)).sum())
ileg_sem_defeito = int((com_med & (n_def == 0) & (rot["legivel"] == 0)).sum())

print(f"fotos com >= 2 defeitos simultâneos: {n_2def} de {len(rot)}")
print("coocorrência de defeitos (diagonal = total de cada defeito):")
display(cooc)
print(f"linhas foto×rotulador sem exatamente uma cena válida: {n_multicena} (o app obriga uma)")
print(f"fotos com defeito marcado e ainda assim legíveis: {leg_com_defeito}")
print(f"fotos com medidor, sem defeito e ilegíveis: {ileg_sem_defeito}")

tab_cabecas = pd.DataFrame([
    {"cabeça": "cena", "saídas": 4, "ativação (inferência)": "softmax", "perda": "CrossEntropyLoss(weight ∝ 1/freq)",
     "exclusivas?": f"sim: toda linha tem exatamente 1 cena ({n_multicena} exceções)"},
    {"cabeça": "defeitos", "saídas": 5, "ativação (inferência)": "sigmoide por defeito",
     "perda": "BCEWithLogitsLoss(pos_weight = neg/pos)",
     "exclusivas?": f"não: {n_2def} fotos com 2+ defeitos ao mesmo tempo"},
    {"cabeça": "legível", "saídas": 1, "ativação (inferência)": "sigmoide", "perda": "BCEWithLogitsLoss",
     "exclusivas?": f"binária; não é função dos defeitos ({leg_com_defeito} legíveis com defeito, "
                    f"{ileg_sem_defeito} ilegíveis sem defeito)"},
])
display(tab_cabecas)
RESUMO["b3"] = {"fotos_2mais_defeitos": n_2def, "legiveis_com_defeito": leg_com_defeito,
                "ilegiveis_sem_defeito_com_medidor": ileg_sem_defeito}

**Decisão sobre a 3ª cabeça (legível): usar.** O plano 04 tratava `legível` como derivado
(`legível = nenhum defeito`), mas o rotulador pergunta separadamente "consigo ler todos os
dígitos?", e os contadores acima mostram fotos com defeito que continuam legíveis (reflexo fora
dos dígitos) e fotos sem defeito marcado que não são legíveis. Uma regra sobre os defeitos erraria
essas fotos por construção; uma cabeça própria custa 577 parâmetros e é exatamente a saída que
alimenta a decisão ACEITA/REJEITADA/DÚVIDA (E2). Os defeitos continuam como **motivo** da recusa.

## 3. D — Sanidade e baseline

Preparação: alvos por conjunto, features congeladas da MobileNetV3 (480 × 360 e 224 letterbox) e
features V0, todas com cache em `lab-02/cache/` (gitignored; só vetores, nenhuma imagem).

In [ ]:
caminho = dict(zip(manifesto["arquivo"], manifesto["caminho"]))
img_fn = modelo.imagem_de_arquivo(caminho)
R = {s: rot[rot["split"] == s].reset_index(drop=True) for s in particao.ORDEM}
Y = {s: modelo.alvos_de_rotulos(R[s]) for s in particao.ORDEM}
for s in particao.ORDEM:
    print(f"{s:<10} {len(R[s]):>4} fotos rotuladas")

t = time.time()
extrator = modelo.ModeloTriagem(pretreinado=True)
todas_rot = list(rot["nome_arquivo"])
for res in ["nativa", "224"]:
    modelo.extrair_features(extrator, todas_rot, img_fn, res, DEVICE, CACHE / f"feat_{res}.npz")
FEAT = {res: {s: modelo.extrair_features(extrator, list(R[s]["nome_arquivo"]), img_fn, res, DEVICE,
                                         CACHE / f"feat_{res}.npz") for s in particao.ORDEM}
        for res in ["nativa", "224"]}
v0_rot = heuristicas.features_v0_lote(todas_rot, [caminho[a] for a in todas_rot], CACHE / "v0.parquet")
V0 = {s: heuristicas.transformar_v0(v0_rot.loc[R[s]["nome_arquivo"]]) for s in particao.ORDEM}
print(f"features prontas em {time.time() - t:.1f} s (device {DEVICE})")

### D1. Perda inicial com pesos aleatórios

In [ ]:
modelo.semear(SEED)
lote_d1 = R["treino"].head(32)
x_d1 = torch.stack([modelo.preprocessar(img_fn(a)) for a in lote_d1["nome_arquivo"]])
y_d1 = modelo.alvos_de_rotulos(lote_d1)
linhas = []
for nome, pre in [("pesos aleatórios (backbone + cabeças)", False), ("backbone ImageNet + cabeças novas", True)]:
    mm = modelo.ModeloTriagem(pretreinado=pre).eval()
    with torch.no_grad():
        out = mm(x_d1)
    yc = torch.tensor(y_d1["cena"])
    ok = yc >= 0
    ce = torch.nn.functional.cross_entropy(out["cena"][ok], yc[ok]).item()
    yd = torch.tensor(y_d1["defeitos"]); okd = ~torch.isnan(yd)
    bce_d = torch.nn.functional.binary_cross_entropy_with_logits(out["defeitos"][okd], yd[okd]).item()
    yl = torch.tensor(y_d1["legivel"]); okl = ~torch.isnan(yl)
    bce_l = torch.nn.functional.binary_cross_entropy_with_logits(out["legivel"][okl], yl[okl]).item()
    linhas.append({"modelo": nome, "CE cena": ce, "ln 4": math.log(4), "BCE defeitos": bce_d,
                   "BCE legível": bce_l, "ln 2": math.log(2)})
d1 = pd.DataFrame(linhas)
display(d1)
assert abs(d1.loc[0, "CE cena"] - math.log(4)) < 0.05 and abs(d1.loc[0, "BCE defeitos"] - math.log(2)) < 0.05
RESUMO["d1"] = d1.iloc[0][["CE cena", "BCE defeitos", "BCE legível"]].round(4).to_dict()

Com a última camada inicializada pequena (σ = 0,01, como o classificador da própria MobileNetV3), os logits começam ≈ 0: a CE fica em ln 4 e cada BCE em ln 2, em 32 fotos reais do treino. Uma perda inicial longe disso apontaria softmax antes da perda ou inicialização errada.

### D2. Sobreajuste de um lote de fotos reais (modelo inteiro destravado, sem dropout)

In [ ]:
N_D2 = min(16, len(R["treino"]))
lote_d2 = R["treino"].sample(N_D2, random_state=SEED).reset_index(drop=True)
img_d2 = torch.stack([torch.from_numpy(img_fn(a)).permute(2, 0, 1) for a in lote_d2["nome_arquivo"]])
y_d2 = modelo.alvos_de_rotulos(lote_d2)
modelo.semear(SEED)
m_d2 = modelo.ModeloTriagem(pretreinado=True, p_dropout=0.0).to(DEVICE).train()
opt = torch.optim.Adam(m_d2.parameters(), lr=1e-3)
media = torch.tensor(modelo.MEDIA, device=DEVICE)[:, None, None]
desvio = torch.tensor(modelo.DESVIO, device=DEVICE)[:, None, None]
x = (img_d2.to(DEVICE).float() / 255 - media) / desvio
yt = {k: torch.tensor(v).to(DEVICE) for k, v in y_d2.items()}
hist_d2 = []
for passo in range(300):
    opt.zero_grad()
    perda = modelo.perda_triagem(m_d2(x), yt)
    perda.backward()
    opt.step()
    hist_d2.append(perda.item())
    if hist_d2[-1] < 0.01:
        break
print(f"{N_D2} fotos reais | perda: passo 0 = {hist_d2[0]:.4f} -> passo {len(hist_d2) - 1} = {hist_d2[-1]:.4f}")
with torch.no_grad():  # mesmo modo do treino (BN com estatística do lote)
    out = m_d2(x)
acerto_cena = (out["cena"].argmax(1).cpu().numpy() == y_d2["cena"])[y_d2["cena"] >= 0].mean()
print(f"acerto de cena nas próprias {N_D2} fotos: {acerto_cena:.0%}")
fig, ax = plt.subplots(figsize=(6, 3))
ax.semilogy(hist_d2)
ax.set_xlabel("passo"); ax.set_ylabel("perda (log)"); ax.set_title(f"D2: sobreajuste de {N_D2} fotos reais")
plt.show()
assert hist_d2[-1] < 0.05, "não decorou o lote: defeito no modelo, na perda ou nos rótulos"
RESUMO["d2"] = {"n_fotos": N_D2, "perda_inicial": round(hist_d2[0], 4), "perda_final": round(hist_d2[-1], 5),
                "passos": len(hist_d2)}
del m_d2, opt, x

A perda vai a ~0 com o modelo inteiro destravado e sem dropout: a rede, a perda mascarada e os alvos estão ligados certo. (O BatchNorm fica em modo treino durante o sobreajuste, como no esqueleto.)

### D3. Extração de características × baseline não profunda

- **V1 (extração):** features congeladas da MobileNetV3 a 480 × 360; só as três cabeças são
  treinadas (AdamW lr 1e-3, weight decay 1e-2, 60 épocas fixas, `pos_weight`/peso de classe; nada escolhido na validação).
- **Baselines V0:** regressão logística e floresta aleatória sobre as 12 features V0
  (`heuristicas.py`: Laplaciano, histograma, brilho, saturação, pixels estourados…).
- Mesma validação (lote 22/05), mesmas métricas: **macro-F1 de cena**, **F1 por defeito e sua
  média (macro-F1 de defeitos)**, **AUROC e F1 de legível**. Três sementes, média ± desvio.

In [ ]:
COLS = ["cena_macro_f1", "defeitos_macro_f1", "legivel_auroc", "legivel_f1"] + [f"f1_{d}" for d in DEFEITOS]

def rodar_v1(res, sementes=SEMENTES, treino=None, tarefas=modelo.TAREFAS, aval="validacao"):
    xtr, ytr = (FEAT[res]["treino"], Y["treino"]) if treino is None else treino
    linhas, modelos = [], []
    for s in sementes:
        cab, norm = modelo.treinar_cabecas(xtr, ytr, seed=s, tarefas=tarefas)
        pred = modelo.prever_cabecas(cab, norm, FEAT[res][aval])
        linhas.append({"semente": s, **avaliacao.metricas(pred, Y[aval], ytr)})
        modelos.append((cab, norm))
    return linhas, modelos

def rodar_v0(tipo, sementes=SEMENTES, treino=None, aval="validacao"):
    xtr, ytr = (V0["treino"], Y["treino"]) if treino is None else treino
    linhas, modelos = [], []
    for s in sementes:
        prever = avaliacao.treinar_baseline(xtr, ytr, tipo, seed=s)
        linhas.append({"semente": s, **avaliacao.metricas(prever(V0[aval]), Y[aval], ytr)})
        modelos.append(prever)
    return linhas, modelos

t = time.time()
l_v1, MOD_V1 = rodar_v1("nativa")
l_lr, MOD_V0LR = rodar_v0("logistica")
l_rf, _ = rodar_v0("floresta")
d3_linhas = ([{"modelo": "V1 extração (MobileNetV3 480×360)", **r} for r in l_v1]
             + [{"modelo": "V0 + regressão logística", **r} for r in l_lr]
             + [{"modelo": "V0 + floresta aleatória", **r} for r in l_rf])
d3 = avaliacao.resumir_sementes(d3_linhas, COLS)
print(f"treino + avaliação em {time.time() - t:.1f} s | validação = lote 22/05, {len(R['validacao'])} fotos rotuladas")
display(d3.set_index("modelo").T)
d3_num = pd.DataFrame(d3_linhas).groupby("modelo", sort=False)[COLS].mean()
RESUMO["d3_validacao"] = d3.set_index("modelo")[["cena_macro_f1", "defeitos_macro_f1", "legivel_auroc"]].to_dict("index")

In [ ]:
v1n, lrn = d3_num.loc["V1 extração (MobileNetV3 480×360)"], d3_num.loc["V0 + regressão logística"]
rfn = d3_num.loc["V0 + floresta aleatória"]
for c in ["cena_macro_f1", "defeitos_macro_f1", "legivel_auroc"] + [f"f1_{d}" for d in DEFEITOS]:
    melhor_v0 = np.nanmax([lrn[c], rfn[c]])
    if np.isnan(v1n[c]) or np.isnan(melhor_v0):
        veredito = "n/d (exemplos insuficientes)"
    else:
        veredito = "V1 ganha" if v1n[c] > melhor_v0 else "V0 ganha"
    print(f"{c:<24} V1 {v1n[c]:.3f} | melhor V0 {melhor_v0:.3f} -> {veredito}")

**Leitura.** A linha de cada métrica acima diz quem ganhou. A regressão logística é determinística
(desvio 0 entre sementes é esperado); a floresta e a cabeça V1 variam com a semente. Onde a V0 ganhar,
as causas prováveis, em ordem, são:
1. **o defeito é um sinal que a V0 mede diretamente** — `fora_de_foco` é alta frequência e o
   Laplaciano é literalmente esse número, enquanto as features do ImageNet foram treinadas para
   serem *invariantes* a borrão e brilho (aumento de dados do próprio ImageNet);
2. **o pool global dilui o display** — o vetor de 576 é a média de 15 × 12 células, e o reflexo
   que cobre o display ocupa poucas delas;
3. **poucos positivos por defeito** (ver a tabela de distribuição): com dezenas de positivos, 5.770
   pesos da cabeça sobreajustam mais que 13 coeficientes da logística.

Onde a V1 ganhar (tipicamente cena, que é semântica), a vantagem vem do pré-treino: distinguir
display LCD de roletes e de fachada é o tipo de padrão que o ImageNet já ensinou.

## 4. E — Confiança e recusa

Modelo V1 = cabeça da semente 0 do D3 (480 × 360). Uma temperatura por saída (cena: softmax;
legível: sigmoide), ajustada por NLL **só na validação**.

### E1. Diagrama de confiabilidade e ECE, antes e depois do *temperature scaling*

In [ ]:
cab0, norm0 = MOD_V1[0]
LOG = {s: modelo.prever_cabecas(cab0, norm0, FEAT["nativa"][s]) for s in ["validacao", "teste"]}
yv = Y["validacao"]
okc = yv["cena"] >= 0
okl = ~np.isnan(yv["legivel"])
T_cena = calibracao.ajustar_temperatura(LOG["validacao"]["cena"][okc], yv["cena"][okc], "softmax")
T_leg = calibracao.ajustar_temperatura(LOG["validacao"]["legivel"][okl], yv["legivel"][okl], "sigmoide")
print(f"T ajustado na validação: cena = {T_cena:.3f} | legível = {T_leg:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
e1 = []
for ax, (nome, conf_fn, z, y, T) in zip(axes, [
        ("cena (top-1, softmax)", calibracao.confianca_multiclasse, LOG["validacao"]["cena"][okc], yv["cena"][okc], T_cena),
        ("legível (sigmoide)", calibracao.confianca_binaria, LOG["validacao"]["legivel"][okl], yv["legivel"][okl].astype(int), T_leg)]):
    c0, a0 = conf_fn(z, y, 1.0)
    c1, a1 = conf_fn(z, y, T)
    calibracao.diagrama_confiabilidade(ax, c0, a0, titulo=f"{nome} — validação", rotulo="antes (T=1)")
    calibracao.diagrama_confiabilidade(ax, c1, a1, titulo=f"{nome} — validação", rotulo=f"depois (T={T:.2f})")
    e1.append({"saída": nome, "n": len(y), "ECE antes": calibracao.ece(c0, a0), "ECE depois": calibracao.ece(c1, a1), "T": T})
plt.tight_layout(); plt.show()
e1 = pd.DataFrame(e1)
display(e1)
RESUMO["e1"] = e1.set_index("saída")[["ECE antes", "ECE depois", "T"]].round(4).to_dict("index")

T > 1 significa que a cabeça saía confiante demais (o `pos_weight` e o treino até convergir empurram os logits); T < 1, tímida demais. O argmax não muda com T; muda só o quanto a probabilidade promete. Os números do diagrama são as fotos por bin: com ~100 fotos na validação, cada bin tem poucas e o ECE tem variância alta.

### E2. Curva cobertura × risco da saída de decisão

Saída de decisão (sobre `p = P(legível)` calibrada; `confiança = max(p, 1 − p)`):

| condição | decisão |
|---|---|
| confiança ≥ limiar e p ≥ 0,5 | **ACEITA** (a foto comprova a leitura) |
| confiança ≥ limiar e p < 0,5 | **REJEITADA** (ilegível: defeito, cena de ocorrência ou não é medidor) |
| resto | **DÚVIDA** (fila humana) |

Cobertura = fração decidida (ACEITA + REJEITADA); risco = taxa de erro entre as decididas.

**Meta do grupo:** cobertura ≥ 30% com risco ≤ 2%. O 30% é o piso de "decidir 30–40% do volume"
(doc 03, seção 9.7). O alvo de produção é mais duro (precisão ≥ 0,995 na auto-aprovação, plano 04
seção 5), mas com ~100 fotos rotuladas por conjunto um erro de 0,5% **não é demonstrável**: zero
erros em n decididas só sustenta risco < 3/n (regra dos três; ~2% com 150 decididas). Por isso o
Y deste laboratório é 2%, e o limite superior de 95% (Clopper-Pearson) é reportado junto.

In [ ]:
def conf_acerto_leg(logits, y, T):
    ok = ~np.isnan(y)
    p = 1 / (1 + np.exp(-logits[ok] / T))
    return np.maximum(p, 1 - p), ((p >= 0.5).astype(int) == y[ok]).astype(float), p, y[ok].astype(int)

cv, av, pv, yv_leg = conf_acerto_leg(LOG["validacao"]["legivel"], Y["validacao"]["legivel"], T_leg)
escolha_v1 = seletiva.escolher_limiar(cv, av, RISCO_META)
cob, risco, _ = seletiva.curva_cobertura_risco(cv, av)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.step(cob, risco, where="post", label="V1 (validação)")
ax.fill_between([COBERTURA_META, 1], 0, RISCO_META, color="green", alpha=0.12, label="região da meta")
ax.plot(COBERTURA_META, RISCO_META, "k*", ms=14, label=f"meta: cob ≥ {COBERTURA_META:.0%}, risco ≤ {RISCO_META:.0%}")
if np.isfinite(escolha_v1["limiar"]):
    ax.plot(escolha_v1["cobertura"], escolha_v1["risco"], "ro", label=f"limiar escolhido = {escolha_v1['limiar']:.3f}")
ax.set_xlabel("cobertura (fração decidida)"); ax.set_ylabel("risco (erro nas decididas)")
ax.set_xlim(0, 1); ax.set_ylim(0, max(0.2, float(np.nanmax(risco)) if len(risco) else 0.2))
ax.set_title("E2: cobertura × risco — saída legível"); ax.legend(fontsize=8)
plt.show()

cabe = escolha_v1["cobertura"] >= COBERTURA_META
if np.isfinite(escolha_v1["limiar"]):
    print(f"validação: limiar = {escolha_v1['limiar']:.4f} -> cobertura {escolha_v1['cobertura']:.1%}, "
          f"risco {escolha_v1['risco']:.2%}")
else:
    print(f"validação: nenhum limiar deixa o risco <= {RISCO_META:.0%}; tudo vai para DÚVIDA (cobertura 0%).")
print("a meta CABE na curva da validação." if cabe else
      "a meta NÃO cabe na curva da validação: com risco ≤ Y a cobertura máxima fica abaixo de 30%.")

In [ ]:
# Teste (03/07): limiar e T da validação, aplicados UMA vez.
ct, at, pt, yt_leg = conf_acerto_leg(LOG["teste"]["legivel"], Y["teste"]["legivel"], T_leg)
res_teste = seletiva.aplicar_limiar(ct, at, escolha_v1["limiar"])
dec = seletiva.decidir(pt, escolha_v1["limiar"])
tab_dec = pd.crosstab(pd.Series(dec, name="decisão"), pd.Series(np.where(yt_leg == 1, "legível", "ilegível"), name="verdade"))
display(tab_dec)
fn_aceita = int(((dec == "ACEITA") & (yt_leg == 0)).sum())
fp_rejeita = int(((dec == "REJEITADA") & (yt_leg == 1)).sum())
print(f"TESTE: cobertura {res_teste['cobertura']:.1%} ({res_teste['decididas']}/{res_teste['n']}), "
      f"risco {res_teste['risco']:.2%} ({res_teste['erros']} erros), limite sup. 95% {res_teste['risco_sup95']:.2%}")
print(f"  ACEITA errada (foto ilegível auto-aprovada): {fn_aceita} | REJEITADA errada (foto boa recusada): {fp_rejeita}")
c_t, a_t = calibracao.confianca_binaria(LOG["teste"]["legivel"][~np.isnan(Y["teste"]["legivel"])],
                                        Y["teste"]["legivel"][~np.isnan(Y["teste"]["legivel"])].astype(int), T_leg)
print(f"  ECE de legível no teste com o T da validação: {calibracao.ece(c_t, a_t):.3f}")
RESUMO["e2"] = {"meta": f"cobertura >= {COBERTURA_META:.0%} com risco <= {RISCO_META:.0%}",
                "limiar_validacao": escolha_v1["limiar"], "cobertura_validacao": escolha_v1["cobertura"],
                "risco_validacao": escolha_v1["risco"], "cobertura_teste": res_teste["cobertura"],
                "risco_teste": res_teste["risco"], "risco_teste_sup95": res_teste["risco_sup95"],
                "aceita_errada_teste": fn_aceita, "rejeitada_errada_teste": fp_rejeita}

Leitura: o ponto vermelho é o que vai para produção — escolhido na validação, conferido no teste. Se o risco do teste passar de Y, o limite superior de 95% mostra quanto da diferença cabe no acaso de uma amostra pequena; o erro que mais importa é ACEITA errada (fatura sem evidência válida).

## 5. K1 — Resolução de entrada: nativa 480 × 360 × 224 (letterbox 3:4)

224 = a foto vai a 224 × 168 sem distorcer e as faixas laterais recebem a média do ImageNet.
Latência do modelo completo (espinha + 3 cabeças), lote de 1 foto, 50 medições após 10 de
aquecimento. CPU com 1 *thread* (no M5, 1 *thread* foi mais rápido que 4 ou 8 para lote 1;
com 4 *workers* de 1 *thread* cada é a configuração do plano 04).

In [ ]:
torch.set_num_threads(1)
m_lat = modelo.ModeloTriagem(pretreinado=True).eval()
k1_lat = []
for res, shape in [("nativa 480×360", (1, 3, 480, 360)), ("224 letterbox", (1, 3, 224, 224))]:
    for dev in [torch.device("cpu")] + ([torch.device("mps")] if DEVICE.type == "mps" else []):
        m_lat.to(dev)
        r = modelo.medir_latencia(m_lat, torch.zeros(shape), dev)
        k1_lat.append({"entrada": res, "device": dev.type, **r})
    # vazão em lote (64) no device de treino
    m_lat.to(DEVICE)
    r = modelo.medir_latencia(m_lat, torch.zeros((64,) + shape[1:]), DEVICE, n=10, aquecimento=3)
    k1_lat.append({"entrada": res, "device": f"{DEVICE.type} lote 64 (por foto)",
                   **{k: v / 64 for k, v in r.items()}})
torch.set_num_threads(NT)
k1_lat = pd.DataFrame(k1_lat)
display(k1_lat)
t0 = time.perf_counter()
for a in R["validacao"]["nome_arquivo"].head(50):
    modelo.preprocessar(img_fn(a))
print(f"decodificar JPEG + normalizar: {(time.perf_counter() - t0) / 50 * 1000:.1f} ms/foto (CPU)")
RESUMO["k1_latencia"] = k1_lat.round(2).to_dict("records")

In [ ]:
l_224, _ = rodar_v1("224")
k1 = avaliacao.resumir_sementes(
    [{"modelo": "V1 extração 480×360 (nativa)", **r} for r in l_v1] + [{"modelo": "V1 extração 224 letterbox", **r} for r in l_224],
    ["cena_macro_f1", "defeitos_macro_f1", "legivel_auroc", "f1_fora_de_foco", "f1_reflexo", "f1_tampa_suja"])
display(k1.set_index("modelo").T)
k1_num = pd.DataFrame([{"modelo": "nativa", **r} for r in l_v1] + [{"modelo": "224", **r} for r in l_224]).groupby("modelo")[COLS].mean()
delta = (k1_num.loc["nativa"] - k1_num.loc["224"])[["cena_macro_f1", "defeitos_macro_f1", "legivel_auroc", "f1_fora_de_foco"]]
print("nativa − 224 (média de 3 sementes):", delta.round(3).to_dict())
lat_cpu = k1_lat[k1_lat["device"] == "cpu"].set_index("entrada")["p50_ms"]
print(f"custo: p50 CPU nativa {lat_cpu.iloc[0]:.1f} ms vs 224 {lat_cpu.iloc[1]:.1f} ms "
      f"({lat_cpu.iloc[0] / lat_cpu.iloc[1]:.1f}×)")
RESUMO["k1_metricas"] = k1.set_index("modelo").to_dict("index")

**Decisão K1.** A regra é: fica a nativa se ela ganhar nas métricas de **qualidade**
(`fora_de_foco`, macro-F1 de defeitos, AUROC de legível) por mais que o desvio entre sementes;
se empatar, 224 vence pelo custo. O custo da nativa em CPU está na tabela de latência e cabe com
folga no orçamento do plano 04 (≈ 580 ms/foto com 4 *workers*). A leitura de dígitos não entra
nesta decisão: ela usa o recorte do display em resolução original, nunca a foto reduzida.

## 6. K2 — O rótulo fraco se sustenta?

Treina as cabeças de **qualidade** (defeitos + legível) **só** com degradação sintética de fotos
boas e avalia **só** em fotos reais rotuladas da validação. Compara com as mesmas cabeças treinadas
com os rótulos reais do treino.

**Origem (fotos limpas):** fotos do treino rotuladas com medidor visível, sem defeito e legíveis.
Se forem menos que `N_FONTE_K2`, completa com fotos **não rotuladas** dos lotes de treino cuja
nitidez V0 (variância do Laplaciano) está no quartil superior do treino.

**Regra para foto real já com defeito:** o rótulo final é o OU lógico (`rotulo_apos_degradacao`):
foto real com defeito X que recebe degradação X fica com X = 1 (borrar o que já está borrado não o
torna menos borrado) e legível = 0; e foto com qualquer defeito real **nunca** é usada como "boa"
de origem (`eh_fonte_limpa`). Assim nenhum exemplo sintético diz "sem defeito" sobre uma foto que
tem defeito.

In [ ]:
limpas = [r["nome_arquivo"] for r in R["treino"].to_dict("records") if degradacao.eh_fonte_limpa(r)]
fonte = list(limpas[:N_FONTE_K2])
n_complemento = 0
if len(fonte) < N_FONTE_K2:
    treino_todo = part[(part["split"] == "treino") & ~part["arquivo"].isin(rot["nome_arquivo"])]
    v0_treino = heuristicas.features_v0_lote(list(treino_todo["arquivo"]), list(treino_todo["caminho"]), CACHE / "v0.parquet")
    q75 = v0_treino["lap_var"].quantile(0.75)
    nitidas = v0_treino[v0_treino["lap_var"] >= q75].sample(frac=1, random_state=SEED).index.tolist()
    complemento = nitidas[:N_FONTE_K2 - len(fonte)]
    n_complemento = len(complemento)
    fonte += complemento
print(f"origem K2: {len(limpas)} fotos de treino rotuladas limpas (usadas {min(len(limpas), N_FONTE_K2)}) "
      f"+ {n_complemento} não rotuladas do quartil superior de nitidez = {len(fonte)}")

# a regra aplicada a um rótulo real com defeito (sem mostrar a foto)
exemplo = {"cena": "digital", **{d: 0 for d in DEFEITOS}, "fora_de_foco": 1, "legivel": 0}
print("regra: foto real com fora_de_foco=1 recebe borrão ->", degradacao.rotulo_apos_degradacao(exemplo, ["fora_de_foco"]),
      "| pode ser origem?", degradacao.eh_fonte_limpa(exemplo))

import hashlib
from functools import lru_cache
@lru_cache(maxsize=64)
def variantes(arquivo):
    semente = (int(hashlib.sha256(arquivo.encode()).hexdigest()[:8], 16) + SEED) % (2**31)
    return degradacao.gerar_sintetico(img_fn(arquivo), seed=semente, n_degradadas=2)

chaves_sint = [f"{a}#{k}" for a in fonte for k in range(3)]
# o cache do sintético muda se o código da degradação mudar
VERSAO_SINT = hashlib.sha256(Path(degradacao.__file__).read_bytes()).hexdigest()[:8]
t = time.time()
X_sint = modelo.extrair_features(extrator, chaves_sint, lambda c: variantes(c.split("#")[0])[int(c.split("#")[1])][0],
                                 "nativa", DEVICE, CACHE / f"feat_sint_nativa_s{SEED}_{VERSAO_SINT}.npz")
rot_sint = pd.DataFrame([variantes(c.split("#")[0])[int(c.split("#")[1])][1] for c in chaves_sint])
print(f"{len(chaves_sint)} exemplos sintéticos (1 limpo + 2 degradados por origem) em {time.time() - t:.1f} s")
display(rot_sint[DEFEITOS + ["legivel"]].sum().rename("positivos").to_frame().T)
Y_sint = {"cena": np.full(len(rot_sint), -1), "defeitos": rot_sint[DEFEITOS].to_numpy(np.float32),
          "legivel": rot_sint["legivel"].to_numpy(np.float32)}

In [ ]:
QUAL = ("defeitos", "legivel")
l_real, _ = rodar_v1("nativa", tarefas=QUAL)
l_sint = []
for s in SEMENTES:
    cab, norm = modelo.treinar_cabecas(X_sint, Y_sint, seed=s, tarefas=QUAL)
    l_sint.append(avaliacao.metricas(modelo.prever_cabecas(cab, norm, FEAT["nativa"]["validacao"]), Y["validacao"], Y["treino"]))
k2 = []
for d in DEFEITOS + ["legivel"]:
    for met in (["auroc", "f1"] if d != "legivel" else ["auroc", "f1"]):
        chave = f"{met}_{d}" if d != "legivel" else f"legivel_{met}"
        real = np.array([r[chave] for r in l_real], float)
        sint = np.array([r[chave] for r in l_sint], float)
        k2.append({"saída": d, "métrica (validação real)": met.upper(),
                   "treino real": "n/d" if np.isnan(real).all() else f"{np.nanmean(real):.3f} ± {np.nanstd(real):.3f}",
                   "só sintético": "n/d" if np.isnan(sint).all() else f"{np.nanmean(sint):.3f} ± {np.nanstd(sint):.3f}",
                   "queda (real − sintético)": np.nan if np.isnan(real).all() or np.isnan(sint).all() else np.nanmean(real) - np.nanmean(sint)})
k2 = pd.DataFrame(k2)
display(k2)
RESUMO["k2"] = {"fontes_rotuladas": min(len(limpas), N_FONTE_K2), "fontes_complemento": n_complemento,
                "exemplos_sinteticos": len(chaves_sint),
                "queda": {f"{r['saída']}_{r['métrica (validação real)']}": (None if pd.isna(r["queda (real − sintético)"]) else round(r["queda (real − sintético)"], 3))
                          for _, r in k2.iterrows()}}

In [ ]:
# O sintético se parece com o real? Distribuição dos sinais V0 que medem cada defeito.
v0_sint = pd.DataFrame([heuristicas.features_v0(variantes(c.split("#")[0])[int(c.split("#")[1])][0]) for c in chaves_sint])
v0_val = v0_rot.loc[R["validacao"]["nome_arquivo"]].reset_index(drop=True)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, (d, f, transf) in zip(axes, [("fora_de_foco", "lap_var", np.log1p), ("reflexo", "frac_estourados", lambda v: v)]):
    bins = 30
    ax.hist(transf(v0_val.loc[R["validacao"][d] == 0, f]), bins=bins, alpha=0.5, density=True, label=f"real, {d}=0")
    ax.hist(transf(v0_val.loc[R["validacao"][d] == 1, f]), bins=bins, alpha=0.5, density=True, label=f"real, {d}=1")
    ax.hist(transf(v0_sint.loc[rot_sint[d] == 1, f]), bins=bins, alpha=0.5, density=True, label=f"sintético, {d}=1", histtype="step", lw=2)
    ax.set_title(f"{d}: {'log(1+' + f + ')' if d == 'fora_de_foco' else f}"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Leitura K2.** A coluna *queda* é o preço do rótulo fraco em cada defeito: perto de zero, o
sintético ensinou o defeito; grande, ensinou o artefato. O histograma mostra o porquê de cada
caso: quando a distribuição do sintético cai em cima do defeito real (borrão e nitidez), a rede
tem chance de transferir; quando o sintético mora numa região que o real não ocupa (a mancha
gaussiana de reflexo, a névoa uniforme), a rede aprende a assinatura da transformação. Por isso o
plano mantém o sintético só como **pré-treino** e decide o modelo com rótulo humano (plano 04,
seção 4.2). Limitação declarada: sem caixa do display, reflexo e display apagado são aplicados
numa região central aproximada.

## 7. K3 — Heurística V0 como baseline oficial

Mesma partição, mesmas métricas (tabela do D3) e a curva cobertura × risco de V0 e V1 no mesmo
gráfico. Três versões de V0: o **Laplaciano puro** (uma feature, logística de 1 variável, a regra
mais simples possível), a **logística sobre as 12 features V0** e a **floresta**. A temperatura não
muda a ordem de confiança de uma saída binária, então as curvas são comparáveis sem calibrar a V0.

In [ ]:
i_lap = heuristicas.NOMES_V0.index("lap_var")
prever_lap = avaliacao.treinar_baseline(V0["treino"][:, [i_lap]], Y["treino"], "logistica")
prever_lr = MOD_V0LR[0]
prever_rf = avaliacao.treinar_baseline(V0["treino"], Y["treino"], "floresta", seed=0)
saidas_v0 = {
    "V0 Laplaciano puro": lambda s: prever_lap(V0[s][:, [i_lap]])["legivel"],
    "V0 logística (12 features)": lambda s: prever_lr(V0[s])["legivel"],
    "V0 floresta (12 features)": lambda s: prever_rf(V0[s])["legivel"],
}
fig, ax = plt.subplots(figsize=(7.5, 4.8))
k3 = []
for nome, fn in list(saidas_v0.items()) + [("V1 MobileNetV3 (calibrada)", None)]:
    if fn is None:
        cval, aval_, ctest, atest = cv, av, ct, at
    else:
        cval, aval_, _, _ = conf_acerto_leg(fn("validacao"), Y["validacao"]["legivel"], 1.0)
        ctest, atest, _, _ = conf_acerto_leg(fn("teste"), Y["teste"]["legivel"], 1.0)
    cob, risco, _ = seletiva.curva_cobertura_risco(cval, aval_)
    ax.step(cob, risco, where="post", label=nome, lw=2.2 if fn is None else 1.4)
    esc = seletiva.escolher_limiar(cval, aval_, RISCO_META)
    tst = seletiva.aplicar_limiar(ctest, atest, esc["limiar"])
    k3.append({"modelo": nome, "cobertura val": esc["cobertura"], "risco val": esc["risco"],
               "cobertura teste": tst["cobertura"], "risco teste": tst["risco"], "risco teste sup95": tst["risco_sup95"],
               "risco total val (cob=100%)": risco[-1] if len(risco) else np.nan})
ax.fill_between([COBERTURA_META, 1], 0, RISCO_META, color="green", alpha=0.12, label="região da meta")
ax.plot(COBERTURA_META, RISCO_META, "k*", ms=14)
ax.set_xlabel("cobertura"); ax.set_ylabel("risco"); ax.set_xlim(0, 1); ax.set_ylim(0, 0.5)
ax.set_title("K3: cobertura × risco na validação — V0 × V1 (saída legível)"); ax.legend(fontsize=8)
plt.show()
k3 = pd.DataFrame(k3)
display(k3)
display(d3.set_index("modelo").T)
RESUMO["k3"] = k3.set_index("modelo").round(4).to_dict("index")

Leitura: a curva mais baixa à direita vence — mais fotos decididas com o mesmo risco. A coluna *cobertura val* é quanto cada modelo decide sem passar de Y na validação; *teste* é o mesmo limiar aplicado uma vez em 03/07. Se a V0 alcançar a V1, a heurística fica como V0/fallback barato e a rede precisa justificar o custo na cena (que a V0 não faz).

## 8. K4 — Multitarefa (1 espinha, 2 cabeças) ou dois modelos separados?

Aqui o compartilhamento só muda alguma coisa se a espinha for **ajustada** (com o extrator
congelado, as features são as mesmas e os dois desenhos coincidem). Protocolo: ajuste fino
completo a 480 × 360, AdamW com taxas diferenciadas (extrator 1e-4, cabeças 1e-3), 6 épocas fixas,
3 sementes, nos rótulos do treino; avaliação na validação.

- **Multitarefa:** uma espinha, cabeças de cena + qualidade (defeitos e legível).
- **Separados:** modelo A só cena; modelo B só qualidade. Em produção, duas passadas.

In [ ]:
def carregar_uint8(df):
    return torch.stack([torch.from_numpy(img_fn(a)).permute(2, 0, 1).contiguous() for a in df["nome_arquivo"]])

IMG_TR, IMG_VA = carregar_uint8(R["treino"]), carregar_uint8(R["validacao"])
configs = {"multitarefa (cena+qualidade)": modelo.TAREFAS, "separado: só cena": ("cena",),
           "separado: só qualidade": ("defeitos", "legivel")}
t = time.time()
k4_linhas, hist_k4 = [], {}
for nome, tarefas in configs.items():
    for s in SEMENTES:
        modelo.semear(s)
        mm = modelo.ModeloTriagem(pretreinado=True, tarefas=tarefas)
        hist_k4[(nome, s)] = modelo.ajuste_fino(mm, IMG_TR, Y["treino"], DEVICE, seed=s, epocas=6)
        pred = modelo.prever_imagens(mm, IMG_VA, DEVICE)
        k4_linhas.append({"modelo": nome, "semente": s, **avaliacao.metricas(pred, Y["validacao"], Y["treino"])})
        del mm
print(f"9 ajustes finos em {time.time() - t:.0f} s")
k4 = avaliacao.resumir_sementes(k4_linhas, ["cena_macro_f1", "defeitos_macro_f1", "legivel_auroc", "legivel_f1"])
display(k4.set_index("modelo").T)

In [ ]:
torch.set_num_threads(1)
mt = modelo.ModeloTriagem(pretreinado=True).eval()
sc = modelo.ModeloTriagem(pretreinado=True, tarefas=("cena",)).eval()
sq = modelo.ModeloTriagem(pretreinado=True, tarefas=("defeitos", "legivel")).eval()
x1 = torch.zeros(1, 3, 480, 360)
k4_custo = []
for dev in [torch.device("cpu")] + ([torch.device("mps")] if DEVICE.type == "mps" else []):
    for mod in (mt, sc, sq):
        mod.to(dev)
    lat_mt = modelo.medir_latencia(mt, x1, dev)
    lat_sep = modelo.medir_latencia(lambda x: (sc(x), sq(x)), x1, dev)
    k4_custo.append({"desenho": "multitarefa", "device": dev.type, "parâmetros": modelo.contar_parametros(mt)[0], **lat_mt})
    k4_custo.append({"desenho": "2 modelos separados", "device": dev.type,
                     "parâmetros": modelo.contar_parametros(sc)[0] + modelo.contar_parametros(sq)[0], **lat_sep})
torch.set_num_threads(NT)
k4_custo = pd.DataFrame(k4_custo)
display(k4_custo.style.format({"parâmetros": "{:,}", "p50_ms": "{:.1f}", "p95_ms": "{:.1f}", "media_ms": "{:.1f}"}))

k4n = pd.DataFrame(k4_linhas).groupby("modelo", sort=False)[COLS].mean()
mt_n = k4n.loc["multitarefa (cena+qualidade)"]
print("efeito da multitarefa (multitarefa − separado), média de 3 sementes:")
print(f"  cena macro-F1     : {mt_n['cena_macro_f1'] - k4n.loc['separado: só cena', 'cena_macro_f1']:+.3f}")
print(f"  defeitos macro-F1 : {mt_n['defeitos_macro_f1'] - k4n.loc['separado: só qualidade', 'defeitos_macro_f1']:+.3f}")
print(f"  legível AUROC     : {mt_n['legivel_auroc'] - k4n.loc['separado: só qualidade', 'legivel_auroc']:+.3f}")
RESUMO["k4"] = {"metricas": k4.set_index("modelo").to_dict("index"),
                "custo": k4_custo.round(2).to_dict("records")}
del mt, sc, sq

**Decisão K4.** A multitarefa custa metade dos parâmetros e uma passada só (a tabela de custo
mostra a latência medida). A regra: fica a multitarefa, a menos que uma das cabeças perca mais
que o desvio entre sementes quando compartilha a espinha (transferência negativa). Os deltas acima
dizem se isso aconteceu. Com rótulos de treino na casa das centenas, compartilhar também funciona
como regularização (Caruana, 1997): cada cabeça vê os gradientes da outra.

## 9. Bônus — Curva de dados: vale a pena rotular mais?

Métrica na validação × número de fotos rotuladas no treino (50, 100, 200, todas), subamostra
sorteada por semente (3 sementes), V1 extração × V0 logística.

In [ ]:
n_total = len(R["treino"])
tamanhos = sorted({n for n in CURVA_N if n < n_total} | {n_total})
curva = []
for n in tamanhos:
    for s in SEMENTES:
        idx = np.random.default_rng(s).permutation(n_total)[:n]
        ysub = {k: v[idx] for k, v in Y["treino"].items()}
        cab, norm = modelo.treinar_cabecas(FEAT["nativa"]["treino"][idx], ysub, seed=s)
        curva.append({"modelo": "V1 extração", "n": n, "semente": s,
                      **avaliacao.metricas(modelo.prever_cabecas(cab, norm, FEAT["nativa"]["validacao"]), Y["validacao"], ysub)})
        prever = avaliacao.treinar_baseline(V0["treino"][idx], ysub, "logistica")
        curva.append({"modelo": "V0 logística", "n": n, "semente": s,
                      **avaliacao.metricas(prever(V0["validacao"]), Y["validacao"], ysub)})
curva = pd.DataFrame(curva)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, met in zip(axes, ["cena_macro_f1", "defeitos_macro_f1", "legivel_auroc"]):
    for nome, g in curva.groupby("modelo"):
        agg = g.groupby("n")[met].agg(["mean", "std"])
        ax.errorbar(agg.index, agg["mean"], yerr=agg["std"].fillna(0), marker="o", capsize=3, label=nome)
    ax.set_xlabel("fotos rotuladas no treino"); ax.set_title(met); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
tab_curva = curva.groupby(["modelo", "n"])[["cena_macro_f1", "defeitos_macro_f1", "legivel_auroc"]].agg(["mean", "std"])
display(tab_curva)
RESUMO["curva_dados"] = {f"{m}|{n}": {k: (None if pd.isna(v) else round(float(v), 3)) for k, v in g[["cena_macro_f1", "defeitos_macro_f1", "legivel_auroc"]].mean().items()}
                         for (m, n), g in curva.groupby(["modelo", "n"])}

Leitura: se a curva da V1 ainda sobe entre 200 e todas as fotos, o gargalo é rótulo e mais rotulagem paga; se achatou, o próximo ganho vem de ajuste fino ou de outra entrada, não de mais rótulos. O desvio em n = 50 mostra o quanto um resultado com poucos rótulos depende do sorteio.

## 10. Resumo — números-chave (para README e ARQUITETURA)

In [ ]:
def fmt(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "n/d"
    if isinstance(v, float):
        return f"{v:.3f}"
    return str(v)

lat = k1_lat.set_index(["entrada", "device"])
v1_d3 = d3.set_index("modelo")
linhas = [
    ("rótulos", "fonte", "SINTÉTICOS (desenvolvimento)" if SINTETICO else "humanos (rotulador)"),
    ("rótulos", "fotos rotuladas / rotuladores", f"{len(rot)} / {RESUMO['rotuladores']}"),
    ("rótulos", "rotuladas treino / validação / teste", " / ".join(str(RESUMO["rotuladas_por_conjunto"].get(s, 0)) for s in particao.ORDEM)),
    ("C3", "kappa cena / legível", f"{fmt(RESUMO.get('kappa', {}).get('cena'))} / {fmt(RESUMO.get('kappa', {}).get('legivel'))}"),
    ("C4", "id_externo e medidores em comum entre conjuntos", str(int(prova_base[["id_externo_em_comum", "medidores_em_comum"]].to_numpy().sum()))),
    ("B2", "parâmetros totais / treináveis (extração)", f"{tab_param.loc[1, 'total']:,} / {tab_param.loc[1, 'treináveis']:,}"),
    ("D1", "CE inicial (ln 4 = 1,386) / BCE inicial (ln 2 = 0,693)", f"{RESUMO['d1']['CE cena']:.3f} / {RESUMO['d1']['BCE defeitos']:.3f}"),
    ("D2", f"sobreajuste {N_D2} fotos: perda inicial → final", f"{hist_d2[0]:.3f} → {hist_d2[-1]:.4f} ({len(hist_d2)} passos)"),
]
for mod in v1_d3.index:
    linhas.append(("D3", f"{mod}: cena macro-F1 | defeitos macro-F1 | legível AUROC",
                   f"{v1_d3.loc[mod, 'cena_macro_f1']} | {v1_d3.loc[mod, 'defeitos_macro_f1']} | {v1_d3.loc[mod, 'legivel_auroc']}"))
for _, r in e1.iterrows():
    linhas.append(("E1", f"ECE {r['saída']}: antes → depois (T)", f"{r['ECE antes']:.3f} → {r['ECE depois']:.3f} (T={r['T']:.2f})"))
e2 = RESUMO["e2"]
linhas += [
    ("E2", "meta", e2["meta"]),
    ("E2", "validação: limiar | cobertura | risco", f"{e2['limiar_validacao']:.3f} | {e2['cobertura_validacao']:.1%} | {fmt(e2['risco_validacao'])}"),
    ("E2", "teste: cobertura | risco | limite sup. 95%", f"{e2['cobertura_teste']:.1%} | {fmt(e2['risco_teste'])} | {fmt(e2['risco_teste_sup95'])}"),
    ("E2", "teste: ACEITA errada | REJEITADA errada", f"{e2['aceita_errada_teste']} | {e2['rejeitada_errada_teste']}"),
]
for (ent, dev), r in lat.iterrows():
    linhas.append(("K1", f"latência {ent} {dev}: p50 / p95 (ms)", f"{r['p50_ms']:.1f} / {r['p95_ms']:.1f}"))
for mod, r in k1.set_index("modelo").iterrows():
    linhas.append(("K1", f"{mod}: defeitos macro-F1 | F1 fora_de_foco | legível AUROC",
                   f"{r['defeitos_macro_f1']} | {r['f1_fora_de_foco']} | {r['legivel_auroc']}"))
for _, r in k2[k2["métrica (validação real)"] == "AUROC"].iterrows():
    linhas.append(("K2", f"AUROC {r['saída']}: real | só sintético", f"{r['treino real']} | {r['só sintético']}"))
for _, r in k3.iterrows():
    linhas.append(("K3", f"{r['modelo']}: cobertura val | teste (risco ≤ {RISCO_META:.0%})", f"{r['cobertura val']:.1%} | {r['cobertura teste']:.1%}"))
for _, r in k4.iterrows():
    linhas.append(("K4", f"{r['modelo']}: cena macro-F1 | defeitos macro-F1 | legível AUROC",
                   f"{r['cena_macro_f1']} | {r['defeitos_macro_f1']} | {r['legivel_auroc']}"))
for _, r in k4_custo.iterrows():
    linhas.append(("K4", f"{r['desenho']} {r['device']}: parâmetros | p50 ms", f"{r['parâmetros']:,} | {r['p50_ms']:.1f}"))
for (mname, n), g in curva.groupby(["modelo", "n"]):
    linhas.append(("Bônus", f"{mname} n={n}: legível AUROC", f"{g['legivel_auroc'].mean():.3f} ± {g['legivel_auroc'].std(ddof=0):.3f}"))
resumo = pd.DataFrame(linhas, columns=["item", "métrica", "valor"])
with pd.option_context("display.max_rows", 200, "display.max_colwidth", 120):
    display(resumo)

RESULTADOS.mkdir(parents=True, exist_ok=True)
resumo.to_csv(RESULTADOS / "resumo.csv", index=False)
with open(RESULTADOS / "resumo.json", "w", encoding="utf-8") as f:
    json.dump(RESUMO, f, ensure_ascii=False, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o))
print(f"resumo gravado em {RESULTADOS} | tempo total do notebook: {(time.time() - T_INICIO) / 60:.1f} min")
if SINTETICO:
    print("!!! Lembrete: esta execução usou RÓTULOS SINTÉTICOS. Nenhum número acima é resultado. !!!")